# Phase 2 Benchmark Runner (Colab)
Thin orchestration notebook. Training logic stays in `src/`.


In [ ]:
# Colab setup + repo bootstrap
import os
from google.colab import drive
drive.mount('/content/drive')

!unzip -q -n "/content/drive/MyDrive/PlantVillage/Datasets.zip" -d "/content/"


if not os.path.isdir('/content/Plant-Disease-Detection-CV'):
    !git clone https://github.com/WilliamKyaww/Plant-Disease-Detection-CV.git
else:
    !cd /content/Plant-Disease-Detection-CV && git pull

!pip install -q -r /content/Plant-Disease-Detection-CV/requirements.txt

In [ ]:
# Canonical repo root
from pathlib import Path
import os, sys

repo_root = Path('/content/Plant-Disease-Detection-CV')  # cloned repo path
if not (repo_root / 'src').is_dir():
    raise FileNotFoundError(f"Repo not found at {repo_root}. Run clone cell first.")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Using repo root:', repo_root)
print('Executed in Colab:', os.path.isdir('/content'))

In [ ]:
# Frozen split guard + dry-run benchmark
from pathlib import Path
manifest = Path('results/split_manifests/latest_split_manifest.json')
if manifest.exists():
    print('Using existing frozen splits (no regeneration).')
    print('Manifest:', manifest.resolve())
else:
    !python -m src.prepare_splits

!python -m src.run_phase2_benchmark --dry-run --no-pretrained --batch-size 4 --num-workers 0 --amp


In [ ]:
# Minimal training smoke (1 model, 1 seed, 1 epoch)
!python -m src.run_phase2_benchmark --models resnet18 --seeds 41 --epochs 1 --batch-size 32 --num-workers 2 --scheduler cosine --class-weighting inverse_frequency


In [ ]:
# Clear stale seed_41 artifacts before full rerun
from pathlib import Path
import shutil

repo = Path('/content/Plant-Disease-Detection-CV')
models = ['resnet18', 'resnet50', 'efficientnet_b0', 'vit_small_patch16_224']

removed = []
for model in models:
    run_dir = repo / 'results' / 'phase2' / 'runs' / model / 'seed_41'
    ckpt_dir = repo / 'models' / 'phase2' / model / 'seed_41'
    for target in (run_dir, ckpt_dir):
        if target.exists():
            shutil.rmtree(target)
            removed.append(str(target))

print(f'Removed {len(removed)} stale seed_41 directories:')
for p in removed:
    print(' -', p)

In [ ]:
# Full rerun for seed_41 across all 4 models
!python -m src.run_phase2_benchmark --models resnet18,resnet50,efficientnet_b0,vit_small_patch16_224 --seeds 41 --epochs 30 --batch-size 32 --num-workers 2 --scheduler cosine --class-weighting inverse_frequency --pretrained --amp

In [ ]:
# Full benchmark run
!python -m src.run_phase2_benchmark --models resnet18,resnet50,efficientnet_b0,vit_small_patch16_224 --seeds 41,42,43 --epochs 30 --batch-size 32 --num-workers 2 --scheduler cosine --class-weighting inverse_frequency --pretrained --amp --resume

In [ ]:
# Rebuild full 12-run Phase 2 summary from metrics.json files
from pathlib import Path
import json
from src.phase2_reporting import write_phase2_summary

repo = Path('/content/Plant-Disease-Detection-CV')
runs_root = repo / 'results' / 'phase2' / 'runs'
out_dir = repo / 'results' / 'phase2'

run_rows = []
for metrics_file in sorted(runs_root.glob('*/seed_*/metrics.json')):
    data = json.loads(metrics_file.read_text(encoding='utf-8'))
    run_rows.append({
        'model': data['model'],
        'seed': int(data['seed']),
        'test_accuracy': float(data['test_accuracy']),
        'test_f1_macro': float(data['test_f1_macro']),
        'elapsed_seconds': float(data['elapsed_seconds']),
        'trainable_params': int(data['trainable_params']),
        'model_size_bytes': int(data['model_size_bytes']),
        'metrics_path': str(metrics_file.relative_to(repo)).replace('\\', '/'),
    })

print('Collected runs:', len(run_rows))
if len(run_rows) != 12:
    print('Warning: expected 12 runs (4 models x 3 seeds).')

paths = write_phase2_summary(run_rows=run_rows, out_dir=str(out_dir))
print('Saved:', paths['model_seed_csv'])
print('Saved:', paths['summary_csv'])
print('Saved:', paths['leaderboard_csv'])

In [ ]:
# Export Phase 2 artifacts to a zip and download to your laptop.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

REPO = Path("/content/Plant-Disease-Detection-CV")
OUT_ZIP = Path("/content/phase2_artifacts_export.zip")

# Keep this False unless you explicitly want all model checkpoints too (much larger zip).
INCLUDE_MODELS = False

paths_to_pack = [
    REPO / "results/phase2/model_seed_metrics.csv",
    REPO / "results/phase2/model_summary_mean_std.csv",
    REPO / "results/phase2/leaderboard.csv",
    REPO / "results/phase2/runs",
    REPO / "results/split_manifests/latest_split_manifest.json",
]

if INCLUDE_MODELS:
    paths_to_pack.append(REPO / "models/phase2")

with ZipFile(OUT_ZIP, "w", ZIP_DEFLATED) as zf:
    for p in paths_to_pack:
        if not p.exists():
            print(f"[skip] missing: {p}")
            continue
        if p.is_file():
            zf.write(p, arcname=p.relative_to(REPO))
        else:
            for f in p.rglob("*"):
                if f.is_file():
                    zf.write(f, arcname=f.relative_to(REPO))

print(f"Created: {OUT_ZIP} ({OUT_ZIP.stat().st_size / (1024**2):.2f} MB)")
files.download(str(OUT_ZIP))
